# B6. Xây dựng và chọn đặc trưng

Gộp 7 bảng về **một dòng mỗi hồ sơ** cho cả 307.511 hồ sơ (train và test), theo quy tắc cố định viết sẵn, đúng định nghĩa của
bảng khớp dữ liệu – chính sách. Quy tắc gộp không học gì từ dữ liệu nên áp cho test không làm lộ thông tin.
Quy tắc **loại cột** thì học từ dữ liệu nên chỉ tính trên khách train và **không dùng TARGET**. Đầu ra `ml/data/features.parquet`.

## 0. Nguồn đã tra và cách áp dụng

| Nguồn | Nội dung | Áp dụng |
|---|---|---|
| Home Credit trên diễn đàn Kaggle (57054, 57562) | Ý nghĩa cột; khoản được duyệt không có kỳ đến hạn đầu là không sử dụng; kỳ chưa trả tính tới ngày kết thúc hợp đồng; kỳ phải trả 0 bỏ qua | Định nghĩa khoản đang hoạt động, kỳ trả nợ (bảng khớp mục 4, 7, chốt 19/10/2026) |
| Lời giải hạng 2 (ikiri_DS, Speaker Deck) | Vay / trả kỳ suy ra kỳ hạn và lãi suất | Có `VAY_TRA_KY` |
| Lời giải hạng 5 (deepsense.ai) | Quá hạn gần đây nặng hơn quá hạn lâu năm; lịch sử 96 tháng; chọn đặc trưng theo mức quan trọng của LightGBM | Có đặc trưng thời điểm (số tháng từ lần quá hạn gần nhất, cửa sổ 12 tháng). Không chọn theo mức quan trọng của mô hình (dùng TARGET, phải lồng trong CV) |
| Lời giải hạng 17 (NoxMoon, GitHub) | Gộp theo nhóm con (khoản đang hoạt động, bị từ chối, hồ sơ gần nhất); vay / trả kỳ; tỷ lệ sử dụng hạn mức; ngày quá hạn gần nhất; chênh số trả và số phải trả | Có các nhóm con và tỷ lệ này. Không dùng mô hình con trên từng dòng, GRU (ngoài phạm vi) |
| QĐ-26 Điều 6 (corpus) | Chỉ dùng dữ liệu hồ sơ, CIC, lịch sử tại AT Credit, điểm tham khảo; không dùng giới tính, danh bạ, mạng xã hội | Bỏ `CODE_GENDER`; bỏ 4 cột `OBS/DEF_xx_CNT_SOCIAL_CIRCLE` (số người quen của khách bị quá hạn: thông tin tín dụng của người khác, không thuộc nhóm nào của khoản 1 Điều 6) |
| Kuhn & Johnson (2013), *Applied Predictive Modeling*, mục 3.5 | Loại cột gần như không đổi | Ngưỡng một giá trị ≥ 99,9% (notebook 04); với tỷ lệ TARGET 8% thì nhóm còn lại ≤ 246 hồ sơ train, kỳ vọng ≤ 20 hồ sơ TARGET = 1, quá ít để ước lượng |
| Jiarpakdee và cộng sự (2018), AutoSpearman | Lọc tương quan Spearman không dùng nhãn: trong cặp tương quan cao bỏ cột có tương quan trung bình với các cột khác cao hơn | Dùng cách chọn cột này với ngưỡng 0,9 (giữ ngưỡng của mục 5.7 EDA): mục đích là bỏ cột trùng thông tin, không bỏ thông tin liên quan nhưng khác (AutoSpearman 0,7 sẽ bỏ một trong hai số tiền vay / trả kỳ) |
| O'Brien (2007), *Quality & Quantity* 41: 673–690 | VIF 10 hay 20 tự nó không đủ lý do bỏ biến | VIF chỉ báo cáo; độ ổn định hệ số của LR xử lý bằng điều chuẩn ở B7 |

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
CLEAN_DIR = ROOT / "ml" / "data" / "clean"
OUT = ROOT / "ml" / "data" / "features.parquet"
THANG = 30.44  # số ngày trung bình một tháng
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")


def load(name):
    d = pd.read_parquet(CLEAN_DIR / f"{name}.parquet")
    return d[d.SK_ID_CURR.isin(a.index)].reset_index(drop=True) if name != "application_train" else d


def nhom_no(ngay):  # khoản 1 Điều 10 TT31/2024: < 10 ngày nhóm 1; 10–90 nhóm 2; 91–180 nhóm 3; 181–360 nhóm 4; > 360 nhóm 5
    return pd.cut(ngay, [-np.inf, 9, 90, 180, 360, np.inf], labels=[1, 2, 3, 4, 5]).astype(float)


a = load("application_train").set_index("SK_ID_CURR")
tap = pd.read_csv(ROOT / "ml" / "data" / "split.csv").set_index("SK_ID_CURR").tap.reindex(a.index)
tr = tap.eq("train")
print(f"{len(a):,} hồ sơ: train {tr.sum():,}, test {(~tr).sum():,}")

307,511 hồ sơ: train 246,008, test 61,503


## 1. Bảng hồ sơ (application)

Giữ các cột gốc, trừ `CODE_GENDER` và 4 cột người quen (mục 0). Thay `DAYS_BIRTH`, `DAYS_EMPLOYED` bằng tuổi (năm) và thâm niên (tháng)
cho dễ đọc khi giải thích; thêm các tỷ lệ của bảng khớp mục 4: **PTI** = tiền trả tháng / thu nhập tháng; vay / trả kỳ; vay / giá hàng; vay / thu nhập.

In [2]:
BO = ["CODE_GENDER", "OBS_30_CNT_SOCIAL_CIRCLE", "DEF_30_CNT_SOCIAL_CIRCLE", "OBS_60_CNT_SOCIAL_CIRCLE", "DEF_60_CNT_SOCIAL_CIRCLE"]
X = a.drop(columns=["TARGET", "DAYS_BIRTH", "DAYS_EMPLOYED", *BO])
X["TUOI"] = -a.DAYS_BIRTH / 365.25
X["THAM_NIEN_THANG"] = -a.DAYS_EMPLOYED / THANG  # trống khi hưu trí / không đi làm (FLAG_NOT_EMPLOYED = 1)
X["PTI"] = a.AMT_ANNUITY / a.AMT_INCOME_TOTAL
X["VAY_TRA_KY"] = a.AMT_CREDIT / a.AMT_ANNUITY
X["VAY_GIA_HANG"] = a.AMT_CREDIT / a.AMT_GOODS_PRICE
X["VAY_THU_NHAP"] = a.AMT_CREDIT / a.AMT_INCOME_TOTAL
X.shape

(307511, 120)

## 2. CIC (bureau, bureau_balance)

Khách không có ở bảng nào thì mọi đặc trưng của bảng đó để **trống**: EDA mục 5.8 cho thấy không có CIC rủi ro cao hơn (10,12% so với 7,73%),
ghi 0 sẽ đánh đồng "không có thông tin" với "không quá hạn". Khách có CIC nhưng không có khoản nào thuộc nhóm con thì đếm và tổng là 0 thật.

- Quá hạn hiện tại: số ngày quá hạn lớn nhất của các khoản **không Closed** (QĐ-07, bảng khớp mục 4); tiền trả tháng chỉ cộng khoản Active (định nghĩa DTI).
- Lịch sử theo tháng: STATUS 1–5 (quá hạn từ 1 ngày), 2–5 (trên 30 ngày), 4–5 (nợ xấu); STATUS 1 không tách được nhóm 1/2 nên không gọi là nhóm 2 (bảng khớp mục 7).

In [3]:
b = load("bureau")
g = b.SK_ID_CURR
act, mo = b.CREDIT_ACTIVE == "Active", b.CREDIT_ACTIVE != "Closed"
theo = lambda mask, col, f: b[col].where(mask).groupby(g).agg(f)
B = pd.DataFrame({
    "CIC_SO_KHOAN": g.value_counts(),
    "CIC_SO_KHOAN_HOAT_DONG": act.groupby(g).sum(),
    "CIC_SO_KHOAN_MO_12_THANG": (b.DAYS_CREDIT >= -365).groupby(g).sum(),
    "CIC_THANG_TU_KHOAN_MOI_NHAT": -b.DAYS_CREDIT.groupby(g).max() / THANG,
    "CIC_DU_NO_HOAT_DONG": theo(act, "AMT_CREDIT_SUM_DEBT", "sum"),
    "CIC_SO_TIEN_VAY_HOAT_DONG": theo(act, "AMT_CREDIT_SUM", "sum"),
    "CIC_TRA_HANG_THANG": theo(act, "AMT_ANNUITY", "sum"),  # 68% khoản Active thiếu: cộng được bao nhiêu tính bấy nhiêu (khoản 4 Điều 4 QĐ-04)
    "CIC_QUA_HAN_HIEN_TAI": theo(mo, "CREDIT_DAY_OVERDUE", "max").fillna(0),
    "CIC_TIEN_QUA_HAN_HIEN_TAI": theo(mo, "AMT_CREDIT_SUM_OVERDUE", "sum"),
    "CIC_TIEN_QUA_HAN_LON_NHAT": b.AMT_CREDIT_MAX_OVERDUE.groupby(g).max(),
    "CIC_SO_LAN_GIA_HAN": b.CNT_CREDIT_PROLONG.groupby(g).sum(),
    "CIC_CO_NO_XAU_DA_XU_LY": b.CREDIT_ACTIVE.isin(["Bad debt", "Sold"]).groupby(g).any().astype(int),
})
B["CIC_TY_LE_DU_NO"] = B.CIC_DU_NO_HOAT_DONG / B.CIC_SO_TIEN_VAY_HOAT_DONG.where(lambda v: v > 0)

bb = pd.read_parquet(CLEAN_DIR / "bureau_balance.parquet").merge(b[["SK_ID_BUREAU", "SK_ID_CURR"]], on="SK_ID_BUREAU")
g, st = bb.SK_ID_CURR, bb.STATUS
q1, q30, q90 = st.isin(list("12345")), st.isin(list("2345")), st.isin(list("45"))
BB = pd.DataFrame({
    "CICT_SO_THANG_QH": q1.groupby(g).sum(),
    "CICT_SO_THANG_QH_TREN_30": q30.groupby(g).sum(),
    "CICT_SO_THANG_NO_XAU": q90.groupby(g).sum(),
    "CICT_TY_LE_THANG_QH": q1.groupby(g).sum() / st.isin(list("012345")).groupby(g).sum().where(lambda v: v > 0),
    "CICT_SO_THANG_QH_12_THANG": (q1 & (bb.MONTHS_BALANCE >= -11)).groupby(g).sum(),  # tháng 0 là tháng nộp
    "CICT_THANG_TU_QH_GAN_NHAT": -bb.MONTHS_BALANCE[q1].groupby(g[q1]).max(),
})
del bb, st, q1, q30, q90
print(f"CIC: {len(B):,} khách; CIC theo tháng: {len(BB):,} khách")

CIC: 263,491 khách; CIC theo tháng: 92,231 khách


## 3. Lịch sử tại AT Credit (previous_application)

**Khoản AT Credit đang hoạt động** (bảng khớp mục 4, chốt 19/10/2026): được duyệt, chưa kết thúc; khoản trả góp phải đã giải ngân (có ngày đến hạn đầu),
hạn mức tính cả khi chưa rút (điểm a, b khoản 2 Điều 5 QC-01). Dùng cho tiền trả tháng trong DTI và tổng dư nợ tối đa (trần 100 triệu, cận trên).

In [4]:
p = load("previous_application")
g, st = p.SK_ID_CURR, p.NAME_CONTRACT_STATUS
duyet, tu_choi = st == "Approved", st == "Refused"
chua_ket_thuc = duyet & (p.FLAG_NOT_TERMINATED.eq(1) | (p.DAYS_TERMINATION > 0))
dang = chua_ket_thuc & (p.DAYS_FIRST_DUE.notna() | (p.NAME_CONTRACT_TYPE == "Revolving loans"))
P = pd.DataFrame({
    "AT_SO_HO_SO": g.value_counts(),
    "AT_SO_HO_SO_DUYET": duyet.groupby(g).sum(),
    "AT_SO_HO_SO_TU_CHOI": tu_choi.groupby(g).sum(),
    "AT_TY_LE_TU_CHOI": tu_choi.groupby(g).mean(),
    "AT_TU_CHOI_30_NGAY": (tu_choi & (p.DAYS_DECISION >= -30)).groupby(g).any().astype(int),  # ngưỡng mềm QĐ-05
    "AT_THANG_TU_HO_SO_GAN_NHAT": -p.DAYS_DECISION.groupby(g).max() / THANG,
    "AT_THANG_TU_TU_CHOI_GAN_NHAT": -p.DAYS_DECISION[tu_choi].groupby(g[tu_choi]).max() / THANG,
    "AT_KY_HAN_TB": p.CNT_PAYMENT.where(duyet & (p.NAME_CONTRACT_TYPE != "Revolving loans")).groupby(g).mean(),
    "AT_SO_KHOAN_HOAT_DONG": dang.groupby(g).sum(),
    "AT_TRA_HANG_THANG": p.AMT_ANNUITY.where(dang).groupby(g).sum(),
    "AT_DU_NO_TOI_DA": p.AMT_CREDIT.where(dang).groupby(g).sum(),  # gốc ban đầu khoản trả góp + toàn bộ hạn mức
})
mo_prev = set(p.SK_ID_PREV[chua_ket_thuc])
print(f"{len(P):,} khách; khoản đang hoạt động {dang.sum():,} / được duyệt chưa kết thúc {chua_ket_thuc.sum():,}")

291,057 khách; khoản đang hoạt động 187,417 / được duyệt chưa kết thúc 194,000


## 4. Kỳ trả nợ (installments_payments)

**Kỳ** = (khoản, số kỳ, ngày đến hạn). Một kỳ chia ở nhiều phiên bản lịch thì số phải trả là tổng các phiên bản, lần trả ghi lặp ở mỗi phiên bản tính một lần
(đúng ở 99,98% kỳ nhiều phiên bản, bảng khớp mục 7). Kỳ phải trả 0 bỏ qua.
**Số ngày quá hạn của kỳ** = ngày của lần trả làm tổng đã trả đạt số phải trả − ngày đến hạn (khoản 7 Điều 2 QC-01; sai số 1 đơn vị);
kỳ chưa trả đủ tính tới ngày hợp đồng kết thúc nếu đã kết thúc, tới ngày nộp nếu chưa.
EDA mục 5.9: quá hạn cách ngày nộp trên 3 năm rủi ro ngang chưa từng quá hạn, nên có thêm đặc trưng thời điểm và cửa sổ 12 tháng.

In [5]:
ip = load("installments_payments")
k = ["SK_ID_PREV", "NUM_INSTALMENT_NUMBER", "DAYS_INSTALMENT"]
can = ip.drop_duplicates(k + ["NUM_INSTALMENT_VERSION"]).groupby(k).AMT_INSTALMENT.sum()
can = can[can > 0]
ip = ip.drop_duplicates(k + ["DAYS_ENTRY_PAYMENT", "AMT_PAYMENT"]).sort_values(k + ["DAYS_ENTRY_PAYMENT"])
ip["cum"] = ip.groupby(k).AMT_PAYMENT.cumsum()
ip = ip.join(can.rename("can"), on=k)
ky = can.to_frame("can").join(ip.groupby(k).agg(cur=("SK_ID_CURR", "first"), da_tra=("AMT_PAYMENT", "sum")))
ky["du"] = ip[ip.cum >= ip.can - 1].groupby(k).DAYS_ENTRY_PAYMENT.min()
end = p.set_index("SK_ID_PREV").DAYS_TERMINATION.reindex(ky.index.get_level_values("SK_ID_PREV")).clip(upper=0).fillna(0).values
han = pd.Series(ky.index.get_level_values("DAYS_INSTALMENT"), index=ky.index)
ky["dpd"] = (ky.du.fillna(pd.Series(end, index=ky.index)) - han).clip(lower=0)
ky["som"] = (han - ky.du).clip(lower=0)  # số ngày trả đủ trước hạn
del ip

g, q10 = ky.cur, ky.dpd >= 10
I = pd.DataFrame({
    "KY_SO_KY": g.value_counts(),
    "KY_MAX_NGAY_QH": ky.dpd.groupby(g).max(),
    "KY_SO_KY_QH_10": q10.groupby(g).sum(),
    "KY_SO_KY_QH_90": (ky.dpd > 90).groupby(g).sum(),
    "KY_TY_LE_KY_QH_10": q10.groupby(g).mean(),
    "KY_MAX_NGAY_QH_12_THANG": ky.dpd.where(han >= -365).groupby(g).max(),  # trống: không có kỳ đến hạn trong 12 tháng
    "KY_THANG_TU_QH_10_GAN_NHAT": -han[q10].groupby(g[q10]).max() / THANG,
    "KY_TY_LE_DA_TRA": ky.da_tra.groupby(g).sum() / ky.can.groupby(g).sum(),
    "KY_NGAY_TRA_SOM_TB": ky.som.groupby(g).mean(),
    "KY_SO_KY_CHUA_TRA_DU": ky.du.isna().groupby(g).sum(),
})
print(f"{len(ky):,} kỳ của {len(I):,} khách; kỳ chưa trả đủ {ky.du.isna().sum():,}")
del ky, han, q10

10,953,086 kỳ của 291,640 khách; kỳ chưa trả đủ 5,047


## 5. Khoản trả góp cũ và khoản AT Flex cũ theo tháng (POS_CASH_balance, credit_card_balance)

Dùng `SK_DPD`, không dùng `SK_DPD_DEF` (bảng khớp mục 7). **Đang quá hạn tại AT Credit** = `SK_DPD` ở tháng ghi gần nhất của khoản đang hoạt động
(chưa kết thúc, trạng thái Active ở tháng đó); tháng ghi cuối là −1 hoặc −2 do độ trễ ghi nhận nên không lấy riêng tháng −1.
**Nhóm nợ của khách** = nhóm của khoản xấu nhất giữa CIC và AT Credit (khoản 4 Điều 1 QĐ-21, mốc TT31). Số tiền rút không tách kênh, tỷ lệ sử dụng hạn mức để trống khi hạn mức bằng 0.

In [6]:
def lich_su(d, ten):
    g, q = d.SK_ID_CURR, d.SK_DPD >= 10
    return pd.DataFrame({
        f"{ten}_MAX_NGAY_QH": d.SK_DPD.groupby(g).max(),
        f"{ten}_SO_THANG_QH_10": q.groupby(g).sum(),
        f"{ten}_THANG_TU_QH_10_GAN_NHAT": -d.MONTHS_BALANCE[q].groupby(g[q]).max(),
    })


def cuoi(d):  # tháng ghi gần nhất của khoản đang hoạt động
    l = d.loc[d.groupby("SK_ID_PREV").MONTHS_BALANCE.idxmax()]
    return l[(l.NAME_CONTRACT_STATUS == "Active") & l.SK_ID_PREV.isin(mo_prev)]


pc = load("POS_CASH_balance")
TG = lich_su(pc, "TG")
lp = cuoi(pc)
TG["TG_SO_KY_CON_LAI"] = lp.CNT_INSTALMENT_FUTURE.groupby(lp.SK_ID_CURR).sum().reindex(TG.index).fillna(0)
co_thang = set(pc.SK_ID_CURR)
del pc

cc = load("credit_card_balance")
HM = lich_su(cc, "HM")
g = cc.SK_ID_CURR
HM["HM_TY_LE_SU_DUNG_TB"] = (cc.AMT_BALANCE / cc.AMT_CREDIT_LIMIT_ACTUAL.where(lambda v: v > 0)).groupby(g).mean()
HM["HM_RUT_12_THANG"] = cc.AMT_DRAWINGS_CURRENT.where(cc.MONTHS_BALANCE >= -12).groupby(g).sum()
lc = cuoi(cc)
HM["HM_TY_LE_SU_DUNG_HIEN_TAI"] = lc.AMT_BALANCE.groupby(lc.SK_ID_CURR).sum() / lc.AMT_CREDIT_LIMIT_ACTUAL.groupby(lc.SK_ID_CURR).sum().where(lambda v: v > 0)
co_thang |= set(cc.SK_ID_CURR)
del cc

cur = pd.concat([lp.groupby("SK_ID_CURR").SK_DPD.max(), lc.groupby("SK_ID_CURR").SK_DPD.max()], axis=1).max(axis=1)
AT_QH = cur.reindex(a.index).where(lambda v: v.notna() | ~a.index.isin(co_thang), 0).rename("AT_QUA_HAN_HIEN_TAI")
print(f"trả góp cũ {len(TG):,} khách, AT Flex cũ {len(HM):,} khách; khoản đang hoạt động có tháng ghi: {len(lp):,} trả góp, {len(lc):,} hạn mức")

trả góp cũ 289,444 khách, AT Flex cũ 86,905 khách; khoản đang hoạt động có tháng ghi: 120,307 trả góp, 67,687 hạn mức


## 6. Ghép và kiểm định nghĩa

Đặc trưng chính sách tính sau khi ghép: **DTI** (khoản 1 Điều 4 QĐ-04: khoản đề nghị + khoản AT Credit đang hoạt động + khoản Active ở CIC, phần thiếu không cộng),
**tổng dư nợ tối đa** (khoản 2 Điều 5 QC-01, cận trên), **nhóm nợ của khách**.
Kiểm tra: tính lại các con số của bảng khớp trên **toàn bộ** 307.511 hồ sơ (cùng tập bảng khớp đã đo) từ chính ma trận đặc trưng; khớp thì đặc trưng ML và corpus dùng cùng định nghĩa.

In [7]:
X = X.join([B, BB, P, I, TG, HM, AT_QH])
X["DTI"] = (a.AMT_ANNUITY + X.AT_TRA_HANG_THANG.fillna(0) + X.CIC_TRA_HANG_THANG.fillna(0)) / a.AMT_INCOME_TOTAL
X["TONG_DU_NO_TOI_DA"] = a.AMT_CREDIT + X.AT_DU_NO_TOI_DA.fillna(0)
X["NHOM_NO_KHACH"] = nhom_no(X[["CIC_QUA_HAN_HIEN_TAI", "AT_QUA_HAN_HIEN_TAI"]].max(axis=1))
X["CIC_NHOM_NO"] = nhom_no(X.CIC_QUA_HAN_HIEN_TAI)

ty_le = lambda m: f"{m.sum():,} ({m.mean():.4%})"
truc_tiep = a.NAME_CONTRACT_TYPE.isin(["Cash loans", "Revolving loans"])
kiem = {
    "PTI > 40% (bảng khớp 7.909; 2,5719%)": X.PTI > 0.4,
    "DTI > 60% (15.900; 5,1705%)": X.DTI > 0.6,
    "thâm niên < 6 tháng (3,17%)": X.THAM_NIEN_THANG < 6,
    "bị từ chối trong 30 ngày (1,23%)": X.AT_TU_CHOI_30_NGAY == 1,
    "CIC nợ xấu, giải ngân trực tiếp (1.016)": (X.CIC_NHOM_NO >= 3) & truc_tiep,
    "CIC nhóm 2 (0,64%)": X.CIC_NHOM_NO == 2,
    "AT Credit đang quá hạn >= 10 ngày (1.293)": X.AT_QUA_HAN_HIEN_TAI >= 10,
    "AT Credit đang quá hạn > 90 ngày (882)": X.AT_QUA_HAN_HIEN_TAI > 90,
    "tổng dư nợ tối đa > 100 triệu (45)": X.TONG_DU_NO_TOI_DA * 24 / 1e6 > 100,
}
for name, m in kiem.items():
    print(f"{name:45s} {ty_le(m)}")
print(X.shape)

PTI > 40% (bảng khớp 7.909; 2,5719%)          7,909 (2.5719%)
DTI > 60% (15.900; 5,1705%)                   15,900 (5.1705%)
thâm niên < 6 tháng (3,17%)                   9,752 (3.1713%)
bị từ chối trong 30 ngày (1,23%)              3,769 (1.2256%)
CIC nợ xấu, giải ngân trực tiếp (1.016)       1,016 (0.3304%)
CIC nhóm 2 (0,64%)                            1,962 (0.6380%)
AT Credit đang quá hạn >= 10 ngày (1.293)     1,293 (0.4205%)
AT Credit đang quá hạn > 90 ngày (882)        882 (0.2868%)
tổng dư nợ tối đa > 100 triệu (45)            45 (0.0146%)
(307511, 175)


## 7. Mức thiếu theo nhóm đặc trưng (train)

Không loại cột vì thiếu: EDA cho thấy thiếu không hoàn toàn ngẫu nhiên ở mọi bảng, ô trống mang thông tin. LightGBM xử lý ô trống trực tiếp;
LR và RF điền giá trị kèm cờ thiếu trong pipeline của B7 (học trong từng fold).

In [8]:
nhom = X.columns.to_series().map(lambda c: c.split("_")[0] if c.split("_")[0] in {"CIC", "CICT", "AT", "KY", "TG", "HM"} else "hồ sơ")
thieu = X[tr].isna().mean()
pd.DataFrame({"số cột": nhom.value_counts(), "thiếu trung vị": thieu.groupby(nhom).median(), "thiếu lớn nhất": thieu.groupby(nhom).max()})

,số cột,thiếu trung vị,thiếu lớn nhất
AT,12,0.0536,0.6734
CIC,14,0.1433,0.4027
CICT,6,0.7006,0.8993
HM,6,0.7182,0.9865
KY,10,0.0516,0.7742
TG,4,0.0587,0.9149
hồ sơ,123,0.0022,0.6984


## 8. Quy tắc loại cột (chỉ train, không dùng TARGET)

1. **Gần như không đổi**: một giá trị (kể cả ô trống) chiếm ≥ 99,9% hồ sơ train.
2. **Trùng thông tin**: Spearman |ρ| ≥ 0,9 giữa hai cột số, tính trên 50.000 hồ sơ train chọn ngẫu nhiên (seed 42), từng cặp chỉ trên các hồ sơ cả hai cột đều có giá trị.
   Lặp: lấy cặp có |ρ| lớn nhất; bỏ cột có |ρ| trung bình với các cột còn lại lớn hơn (AutoSpearman), trừ khi một cột là **khái niệm chính sách**
   (bảng khớp mục 4, ML Tool dùng để giải thích) thì giữ cột đó; hai cột đều là khái niệm chính sách thì giữ cả hai.
   Cột chữ: EDA mục 5.7 không có cặp nào có Cramér's V ≥ 0,8, các cột mới đều là số, nên không cần quy tắc riêng.
3. **VIF** trên các cột số còn lại (điền trung vị, cùng mẫu): chỉ báo cáo.

In [9]:
top = X[tr].apply(lambda c: c.value_counts(normalize=True, dropna=False).iloc[0])
bo_hang_so = top[top >= 0.999].sort_values(ascending=False)
print(f"Quy tắc 1: bỏ {len(bo_hang_so)} cột"); print(bo_hang_so.to_string())
F = X.drop(columns=bo_hang_so.index)

Quy tắc 1: bỏ 10 cột
FLAG_MOBIL         1.0000
FLAG_DOCUMENT_12   1.0000
FLAG_DOCUMENT_10   1.0000
FLAG_DOCUMENT_2    1.0000
FLAG_DOCUMENT_4    0.9999
FLAG_DOCUMENT_7    0.9998
FLAG_DOCUMENT_17   0.9998
FLAG_DOCUMENT_21   0.9997
FLAG_DOCUMENT_20   0.9995
FLAG_DOCUMENT_19   0.9994


In [10]:
BAO_VE = {"AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "TUOI", "THAM_NIEN_THANG", "PTI", "DTI", "TONG_DU_NO_TOI_DA",
          "NHOM_NO_KHACH", "AT_TU_CHOI_30_NGAY", "AT_QUA_HAN_HIEN_TAI", "CIC_QUA_HAN_HIEN_TAI"}
so = F.select_dtypes("number").columns
mau = F.loc[tr].sample(50_000, random_state=42)
r = mau[so].corr(method="spearman").abs()
r = r.where(~np.eye(len(r), dtype=bool))
bo_tuong_quan = []
while True:
    cap = r.stack()
    cap = cap[(cap >= 0.9) & np.array([not (i in BAO_VE and j in BAO_VE) for i, j in cap.index])]
    if cap.empty:
        break
    (i, j), rho = cap.idxmax(), cap.max()
    bo = j if i in BAO_VE else i if j in BAO_VE else (i if r[i].mean() > r[j].mean() else j)
    bo_tuong_quan.append({"bỏ": bo, "giữ": j if bo == i else i, "|rho|": rho})
    r = r.drop(index=bo, columns=bo)
bo_tuong_quan = pd.DataFrame(bo_tuong_quan)
print(f"Quy tắc 2: bỏ {len(bo_tuong_quan)} cột")
F = F.drop(columns=bo_tuong_quan["bỏ"])
bo_tuong_quan

Quy tắc 2: bỏ 41 cột


,bỏ,giữ,|rho|
0,FLAG_NOT_EMPLOYED,FLAG_EMP_PHONE,0.9998
1,YEARS_BUILD_AVG,YEARS_BUILD_MEDI,0.9980
2,YEARS_BEGINEXPLUATATION_AVG,YEARS_BEGINEXPLUATATION_MEDI,0.9969
3,LANDAREA_AVG,LANDAREA_MEDI,0.9968
4,COMMONAREA_AVG,COMMONAREA_MEDI,0.9965
5,LIVINGAPARTMENTS_AVG,LIVINGAPARTMENTS_MEDI,0.9964
6,FLOORSMIN_AVG,FLOORSMIN_MEDI,0.9959
7,LIVINGAREA_AVG,LIVINGAREA_MEDI,0.9956
8,APARTMENTS_AVG,APARTMENTS_MEDI,0.9955
9,FLOORSMAX_MEDI,FLOORSMAX_AVG,0.9951


In [11]:
cap_bao_ve = r.stack()
cap_bao_ve = cap_bao_ve[cap_bao_ve >= 0.9]
print("Cặp |rho| >= 0,9 còn lại (đều là khái niệm chính sách):", {k: round(v, 3) for k, v in cap_bao_ve.items() if k[0] < k[1]})

so = F.select_dtypes("number").columns
m = mau[so].fillna(mau[so].median())
m = m.loc[:, m.std() > 0]
vif = pd.Series(np.diag(np.linalg.pinv(np.corrcoef(m.values, rowvar=False))), index=m.columns).sort_values(ascending=False)
print(f"Quy tắc 3: {len(vif)} cột số; VIF > 10: {(vif > 10).sum()}; VIF > 5: {(vif > 5).sum()}")
vif.head(15)

Cặp |rho| >= 0,9 còn lại (đều là khái niệm chính sách): {}
Quy tắc 3: 109 cột số; VIF > 10: 9; VIF > 5: 19


HM_MAX_NGAY_QH               102.4750
HM_SO_THANG_QH_10            101.2587
DTI                           59.7618
CIC_TRA_HANG_THANG            58.7601
CICT_SO_THANG_QH_TREN_30      24.9164
CICT_SO_THANG_NO_XAU          22.7192
AMT_CREDIT                    19.2749
VAY_THU_NHAP                  16.4239
PTI                           12.7104
AMT_ANNUITY                    9.8045
REG_REGION_NOT_WORK_REGION     8.9952
TG_MAX_NGAY_QH                 8.4269
TG_SO_THANG_QH_10              8.3603
REG_CITY_NOT_WORK_CITY         7.3682
VAY_TRA_KY                     7.3127
dtype: float64

## 9. Lưu

In [12]:
cat = F.select_dtypes(exclude="number").columns
print(f"{F.shape[1]} đặc trưng: {len(F.columns) - len(cat)} số, {len(cat)} chữ ({', '.join(cat)})")
F.insert(0, "TARGET", a.TARGET)
F.reset_index().to_parquet(OUT, index=False)
print(f"{OUT.relative_to(ROOT)}: {len(F):,} dòng; train {tr.sum():,}, test {(~tr).sum():,}")

124 đặc trưng: 109 số, 15 chữ (NAME_CONTRACT_TYPE, FLAG_OWN_CAR, FLAG_OWN_REALTY, NAME_TYPE_SUITE, NAME_INCOME_TYPE, NAME_EDUCATION_TYPE, NAME_FAMILY_STATUS, NAME_HOUSING_TYPE, OCCUPATION_TYPE, WEEKDAY_APPR_PROCESS_START, ORGANIZATION_TYPE, FONDKAPREMONT_MODE, HOUSETYPE_MODE, WALLSMATERIAL_MODE, EMERGENCYSTATE_MODE)


ml/data/features.parquet: 307,511 dòng; train 246,008, test 61,503
